# Deep Convolutional Generative Adversarial Network

여기서는 DCGAN을 작성합니다.
GAN은 오늘날 컴퓨터 과학에서 가장 흥미로운 아이디어 중 하나입니다. 두 가지 모델은 적대적 프로세스에 의해 동시에 학습됩니다. `Generator`는 실제처럼 보이는 이미지를 만드는 법을 배우고, `Discriminator`는 실제 이미지와 가짜 이미지를 구별하는 법을 배웁니다.

훈련이 거듭될수록 실제처럼 보이는 이미지를 생성하는 데 능숙해지고, 구별하는 능력도 능숙해집니다. 여기서는 MNIST데이터셋에서 이 과정을 보여줍니다.
다음 애니메이션은 `Generator`가 50개의 에포크 동안 훈련한 일련의 이미지를 보여줍니다. 이미지는 무작위 노이즈로 시작하여 시간이 지남에 따라 손으로 쓴 숫자와 점점 더 닮아갑니다.

<p align='center'>
<img src='https://tensorflow.org/images/gan/dcgan.gif'>
</img>
</p>

## 0. 환경설정

In [ ]:
!pip install imageio

In [ ]:
import glob, imageio, matplotlib.pyplot as plt, numpy as np, os, time
from tensorflow.keras import layers
from IPython import display

## 1. MNIST 데이터 준비

In [ ]:
from tensorflow.keras.datasets import mnist
(train_images, train_labels), (_, _) = mnist.load_data()
train_images = train_images.reshape(
    train_images.shape[0], 28, 28, 1).astype('float32')
train_images = (train_images - 127.5) / 127.5 # Normalize the images to [-1, 1]

In [ ]:
# 영상 데이터 확인
train_images.shape, type(train_images)

In [ ]:
# 라벨 확인
train_labels.shape, type(train_labels), np.unique(train_labels)

In [ ]:
import tensorflow as tf
# Batch and shuffle the data
train_dataset = tf.data.Dataset.from_tensor_slices(train_images
                    ).shuffle(train_images.shape[0]).batch(256)
# 영상 데이터 batch 확인
next(iter(train_dataset)).shape, type(train_dataset)

## 2. Generator 

`Generator`는 업샘플링 `tf.keras.layers.Conv2Dtranspose`레이어를 사용하여 랜덤 노이즈로부터 이미지를 생성합니다. 이 노이즈를 입력으로 받는 `Dense`레이어로부터 시작하여 원하는 이미지 크기`28x28x1`에 도달할 때까지 여러 번 업샘플링합니다.
`tanh`를 사용하여 출력 레이어를 제외한 각 레이어에 대해 `tf.keras.layers.LeakyReLU` 활성화를 이용합니다.

In [ ]:
# Generator 구현
def make_generator_model():
    model = tf.keras.Sequential()
    # Dense Layer로 시작
    model.add(layers.Dense(7*7*256, use_bias=False, input_shape=(100,)))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())
    # reshape
    model.add(layers.Reshape((7, 7, 256)))
    assert model.output_shape == (None, 7, 7, 256)
    # add Conv2DTranspose layer
    model.add(layers.Conv2DTranspose(128, (5, 5), strides=(1, 1),
                        padding='same', use_bias=False))
    assert model.output_shape == (None, 7, 7, 128)
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())
    # add Conv2DTranspose layer
    model.add(layers.Conv2DTranspose(64, (5, 5), strides=(2, 2),
                        padding='same', use_bias=False))
    assert model.output_shape == (None, 14, 14, 64)
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())
    # add Conv2DTranspose layer
    model.add(layers.Conv2DTranspose(1, (5, 5), strides=(2, 2),
                padding='same', use_bias=False, activation='tanh'))
    assert model.output_shape == (None, 28, 28, 1)
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())
    return model

In [ ]:
generator = make_generator_model()
noise = tf.random.normal([1, 100])
generated_image = generator(noise, training=False)
plt.imshow(generated_image[0, :, :, 0], cmap='gray')

## 3. Discriminator

판별기는 CNN기반 이미지 분류기입니다. `make_discriminator_model()`함수의 시작 부분에서 입력_shape는 생성기의 출력 모양과 같아야 합니다.

In [ ]:
# Generator 구현
def make_discriminator_model():
    model = tf.keras.Sequential()
    # Conv2D Layer로 시작
    model.add(layers.Conv2D(64, (5, 5), strides=(2, 2),
                padding='same', input_shape=[28, 28, 1]))    
    model.add(layers.LeakyReLU())
    model.add(layers.Dropout(.3))
    # Conv2D Layer 추가
    model.add(layers.Conv2D(128, (5, 5), strides=(2, 2),
                padding='same'))    
    model.add(layers.LeakyReLU())
    model.add(layers.Dropout(.3))
    # 결정 neuron 1개 Layer 추가
    model.add(layers.Flatten())
    model.add(layers.Dense(1))
    return model

(아직 훈련되지 않은) 판별기를 사용하여 생성된 이미지를 실제 이미지와 가짜 이미지로 분류합니다. 모델은 실제 이미지의 경우 양수 값을, 가짜 이미지의 경우 음수 값을 출력하도록 훈련됩니다.

### GAN은 어떻게 학습하나요?

노이즈가 있는 데이터 $x \sim p_z(z)$를 사용하여 판별기를 $D(x;\theta_d)$라고 가정합니다. 여기서 $p_z$는 사전 분포이고 생성기는 $G(z; \theta_g)$입니다. 그런 다음 목표는 $G$에서 학습 예제와 샘플 모두에 올바른 레이블을 할당할 확률을 최대화하기 위해 $D$를 학습하는 것입니다. 우리는 동시에 $\log(1-D(z))$를 최소화하도록 $G$를 학습시킵니다.
즉, $D$와 $G$는 목적 함수 $\mathcal{V}(D, G)$ $$\min_G \max_D \mathcal{V}(D, G) = \mathbb{E}_{x \sim p_{\text{data}(x)}} [\log D(x)] + \mathbb{E}_{z \sim p_z(z)} [\log (1 - D(G(z)))]$$ 를 갖는 다음 두 계층 미니맥스 게임을 수행합니다.

각 학습 단계에서 Discriminator의 기울기가 업데이트되고 $$\nabla_{\theta_d} \frac{1}{m} \sum_{i=1}^m [\log D(x^{(i)}) + \log(1 - D(G(z^{(i)})))]$$  Generator의 기울기가 업데이트됩니다. $$\nabla_{\theta_g} \frac{1}{m} \sum_{i=1}^m \log(1 - D(G(z^{(i)})))$$

In [ ]:
discriminator = make_discriminator_model()
decision = discriminator(generated_image)
print(decision)

## 4. 손실함수

In [ ]:
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)
def generator_loss(fake_output):
    return cross_entropy(tf.ones_like(fake_output), fake_output)
# 확인
tf.ones_like([1,2,3]), tf.zeros_like([1,2,3])

In [ ]:
def discriminator_loss(real_output, fake_output):
    real_loss = cross_entropy(tf.ones_like(real_output), real_output)
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)
    total_loss = real_loss + fake_loss
    return total_loss

## 5. optimizer와 save checkpoints

In [ ]:
generator_optimizer = tf.keras.optimizers.Adam(1e-4)
discriminator_optimizer = tf.keras.optimizers.Adam(1e-4)
checkpoint_dir = './training_checkpoints'
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt")
checkpoint = tf.train.Checkpoint(generator_optimizer=generator_optimizer,
                                discriminator_optimizer=discriminator_optimizer,
                                generator=generator, discriminator=discriminator)

## 6. 훈련 loop

In [ ]:
Epochs = 50
noise_dim = 100
num_examples_to_generate = 25
seed = tf.random.normal([num_examples_to_generate, noise_dim])

In [ ]:
@tf.function
def train_step(images):
    # args
    noise = tf.random.normal([256, noise_dim])
    # gradient tape는 역전파를 위한 미분계산 scope
    with tf.GradientTape() as gen_tape, tf.GradientTape as disc_tape:
        generated_images = generator(noise, training=True)
        real_output = discriminator(images, training=True)
        fake_output = discriminator(generated_images, training=True)
        gen_loss = generator_loss(fake_output)    
        disc_loss = discriminator_loss(real_output, fake_output)    
    # compute gradients
    gradient_of_generator = gen_tape.gradient(gen_loss, 
                                              generator.trainable_variables)
    gradient_of_discriminator = disc_tape.gradient(disc_loss, 
    discriminator.trainable_variables)
    # apply gradients
    generator_optimizer.apply_gradients(zip(gradient_of_generator, 
                                            generator.trainable_variables))
    discriminator_optimizer.apply_gradients(zip(gradient_of_discriminator, 
                     discriminator.trainable_variables))

In [ ]:
# GAN